# Crime Risk Analysis Mexico
## 02 — Dashboard Dataset Preparation

This notebook extracts the automobile theft analytical dataset from SQL Server
and performs final data-quality checks before visualization in Power BI.

The dataset focuses specifically on four-wheel automobile theft and separates
incidents into violent and non-violent modalities.

### Objectives

- Extract the analytical view from SQL Server.
- Validate dataset dimensions and data types.
- Check missing and duplicated records.
- Validate the automobile-theft metrics.
- Create a standardized municipality identifier.
- Export a clean analytical dataset.

In [1]:
import pandas as pd
from sqlalchemy import create_engine
from urllib.parse import quote_plus

In [2]:
server = "worklap"
database = "CrimeRiskMexico"
driver = "ODBC Driver 18 for SQL Server"

connection_string = quote_plus(
    f"DRIVER={{{driver}}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    "Trusted_Connection=yes;"
    "TrustServerCertificate=yes;"
)

engine = create_engine(
    f"mssql+pyodbc:///?odbc_connect={connection_string}"
)

In [3]:
query = """
SELECT *
FROM vw_riesgo_automotriz;
"""

df_riesgo = pd.read_sql(query, engine)

df_riesgo.head()

,clave_entidad,entidad,clave_municipio,municipio,anio,numero_mes,mes,fecha,robos_con_violencia,robos_sin_violencia,robos_totales
0,1,Aguascalientes,1001,Aguascalientes,2015,5,mayo,2015-05-01,0,138,138
1,1,Aguascalientes,1001,Aguascalientes,2016,3,marzo,2016-03-01,4,180,184
2,1,Aguascalientes,1001,Aguascalientes,2016,7,julio,2016-07-01,4,202,206
3,1,Aguascalientes,1001,Aguascalientes,2017,3,marzo,2017-03-01,4,183,187
4,1,Aguascalientes,1001,Aguascalientes,2017,6,junio,2017-06-01,5,182,187


In [4]:
print(f"Rows: {df_riesgo.shape[0]:,}")
print(f"Columns: {df_riesgo.shape[1]}")

Rows: 313,836
Columns: 11


In [5]:
df_riesgo.info()

<class 'pandas.DataFrame'>
RangeIndex: 313836 entries, 0 to 313835
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   clave_entidad        313836 non-null  int64 
 1   entidad              313836 non-null  str   
 2   clave_municipio      313836 non-null  int64 
 3   municipio            313836 non-null  str   
 4   anio                 313836 non-null  int64 
 5   numero_mes           313836 non-null  int64 
 6   mes                  313836 non-null  str   
 7   fecha                313836 non-null  object
 8   robos_con_violencia  313836 non-null  int64 
 9   robos_sin_violencia  313836 non-null  int64 
 10  robos_totales        313836 non-null  int64 
dtypes: int64(7), object(1), str(3)
memory usage: 35.5+ MB


In [6]:
df_riesgo["fecha"] = pd.to_datetime(df_riesgo["fecha"])

df_riesgo["fecha"].dtype

dtype('<M8[s]')

In [7]:
null_values = df_riesgo.isnull().sum()

null_values[null_values > 0]

Series([], dtype: int64)

In [8]:
duplicate_rows = df_riesgo.duplicated().sum()

print(f"Duplicate rows: {duplicate_rows:,}")

Duplicate rows: 0


In [9]:
key_duplicates = df_riesgo.duplicated(
    subset=["clave_entidad", "clave_municipio", "fecha"]
).sum()

print(f"Duplicate municipality-month keys: {key_duplicates:,}")

Duplicate municipality-month keys: 0


In [10]:
metric_mismatches = (
    df_riesgo["robos_totales"]
    !=
    df_riesgo["robos_con_violencia"]
    + df_riesgo["robos_sin_violencia"]
).sum()

print(f"Metric inconsistencies: {metric_mismatches:,}")

Metric inconsistencies: 0


In [11]:
metric_columns = [
    "robos_con_violencia",
    "robos_sin_violencia",
    "robos_totales"
]

negative_values = (df_riesgo[metric_columns] < 0).sum()

negative_values

robos_con_violencia    0
robos_sin_violencia    0
robos_totales          0
dtype: int64

In [12]:
df_riesgo["id_municipio"] = (
    df_riesgo["clave_municipio"]
    .astype(str)
    .str.zfill(5)
)

df_riesgo[
    ["clave_entidad", "clave_municipio", "municipio", "id_municipio"]
].head(10)

,clave_entidad,clave_municipio,municipio,id_municipio
0,1,1001,Aguascalientes,01001
1,1,1001,Aguascalientes,01001
2,1,1001,Aguascalientes,01001
3,1,1001,Aguascalientes,01001
4,1,1001,Aguascalientes,01001
5,1,1001,Aguascalientes,01001
6,1,1001,Aguascalientes,01001
7,1,1001,Aguascalientes,01001
8,1,1001,Aguascalientes,01001
9,1,1001,Aguascalientes,01001


In [13]:
print(f"Final rows: {df_riesgo.shape[0]:,}")
print(f"Final columns: {df_riesgo.shape[1]}")

df_riesgo.head()

Final rows: 313,836
Final columns: 12


,clave_entidad,entidad,clave_municipio,municipio,anio,numero_mes,mes,fecha,robos_con_violencia,robos_sin_violencia,robos_totales,id_municipio
0,1,Aguascalientes,1001,Aguascalientes,2015,5,mayo,2015-05-01,0,138,138,01001
1,1,Aguascalientes,1001,Aguascalientes,2016,3,marzo,2016-03-01,4,180,184,01001
2,1,Aguascalientes,1001,Aguascalientes,2016,7,julio,2016-07-01,4,202,206,01001
3,1,Aguascalientes,1001,Aguascalientes,2017,3,marzo,2017-03-01,4,183,187,01001
4,1,Aguascalientes,1001,Aguascalientes,2017,6,junio,2017-06-01,5,182,187,01001
